# DS511: Applied Multivariate Analysis — Course Project Part 1
## Predictive Maintenance of Turbofan Engines | NASA C-MAPSS
**IIT Ropar | MSDSM-03 | AY 2025-2026**

**Group:** Swastayan Borah (2025DSS1028), Vatsal Goswami (2025DSS1031)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Lasso, LassoCV, Ridge, RidgeCV
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True
sns.set_theme(style='whitegrid')

COLS = ['unit', 'cycle',
        'op1', 'op2', 'op3',
        's1','s2','s3','s4','s5','s6','s7','s8','s9','s10',
        's11','s12','s13','s14','s15','s16','s17','s18','s19','s20','s21']

# Sensors with near-zero variance (constant / non-informative) — dropped per literature
DROP_SENSORS = ['s1','s5','s10','s16','s18','s19']
FEATURES = [c for c in COLS[2:] if c not in DROP_SENSORS]  # op1-3 + 15 sensors

print('Feature columns used:', FEATURES)

---
## 1. Data Loading & RUL Labelling

In [ ]:
def load_dataset(fd_id, base='/mnt/user-data/uploads/'):
    """Load train/test/RUL for a given FD subset."""
    train = pd.read_csv(f'{base}train_FD00{fd_id}.txt', sep=' ', header=None,
                        names=COLS, index_col=False).dropna(axis=1)
    test  = pd.read_csv(f'{base}test_FD00{fd_id}.txt',  sep=' ', header=None,
                        names=COLS, index_col=False).dropna(axis=1)
    rul   = pd.read_csv(f'{base}RUL_FD00{fd_id}.txt',   header=None, names=['RUL'])

    # RUL for training: max_cycle - current_cycle
    max_cycles = train.groupby('unit')['cycle'].max().reset_index()
    max_cycles.columns = ['unit', 'max_cycle']
    train = train.merge(max_cycles, on='unit')
    train['RUL'] = train['max_cycle'] - train['cycle']
    train.drop('max_cycle', axis=1, inplace=True)

    # RUL for test: last cycle of each engine + ground truth RUL
    last = test.groupby('unit')['cycle'].max().reset_index()
    last.columns = ['unit', 'max_cycle']
    last['RUL_true'] = rul['RUL'].values
    test = test.merge(last, on='unit')
    test['RUL'] = (test['max_cycle'] - test['cycle']) + test['RUL_true']
    test.drop(['max_cycle', 'RUL_true'], axis=1, inplace=True)

    return train, test

datasets = {}
for fd in [1, 2, 3, 4]:
    tr, te = load_dataset(fd)
    datasets[fd] = {'train': tr, 'test': te}
    print(f'FD00{fd} | train: {tr.shape} | test: {te.shape}')

---
## 2. Preprocessing
### 2a. Outlier Removal — Median ± 3×MAD (per sensor, per unit)

In [ ]:
def remove_outliers_mad(df, sensor_cols, k=3):
    """Per-column median ± k*MAD outlier removal."""
    mask = pd.Series(True, index=df.index)
    for col in sensor_cols:
        median = df[col].median()
        mad = (df[col] - median).abs().median()
        lower = median - k * mad
        upper = median + k * mad
        mask &= df[col].between(lower, upper)
    return df[mask].reset_index(drop=True)

sensor_cols = [c for c in FEATURES if c.startswith('s')]

for fd in [1, 2, 3, 4]:
    n_before = len(datasets[fd]['train'])
    datasets[fd]['train'] = remove_outliers_mad(datasets[fd]['train'], sensor_cols)
    n_after = len(datasets[fd]['train'])
    print(f'FD00{fd}: removed {n_before - n_after} outlier rows ({100*(n_before-n_after)/n_before:.2f}%)')

### 2b. 70/30 Train-Validation Split & Normalisation

In [ ]:
splits = {}
scalers = {}

for fd in [1, 2, 3, 4]:
    df = datasets[fd]['train']
    X = df[FEATURES].values
    y = df['RUL'].values

    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.30, random_state=42)

    scaler = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)

    splits[fd]  = (X_tr_s, X_val_s, y_tr, y_val)
    scalers[fd] = scaler
    print(f'FD00{fd}: train={X_tr_s.shape}, val={X_val_s.shape}')

---
## 3. Feature Selection
### 3a. Forward Selection

In [ ]:
fwd_features = {}

for fd in [1, 2, 3, 4]:
    X_tr, X_val, y_tr, y_val = splits[fd]
    sfs = SequentialFeatureSelector(LinearRegression(), n_features_to_select='auto',
                                    direction='forward', scoring='r2', cv=5, n_jobs=-1)
    sfs.fit(X_tr, y_tr)
    selected_idx = sfs.get_support(indices=True)
    fwd_features[fd] = selected_idx
    sel_names = [FEATURES[i] for i in selected_idx]
    print(f'FD00{fd} Forward Selection ({len(sel_names)} features): {sel_names}')

### 3b. Lasso Feature Selection

In [ ]:
lasso_features = {}

for fd in [1, 2, 3, 4]:
    X_tr, X_val, y_tr, y_val = splits[fd]
    lasso_cv = LassoCV(cv=5, max_iter=10000, random_state=42)
    lasso_cv.fit(X_tr, y_tr)
    coef_mask = lasso_cv.coef_ != 0
    lasso_features[fd] = np.where(coef_mask)[0]
    sel_names = [FEATURES[i] for i in lasso_features[fd]]
    print(f'FD00{fd} Lasso (alpha={lasso_cv.alpha_:.4f}, {len(sel_names)} features): {sel_names}')

### 3c. PCA-Based Feature Selection (95% variance)

In [ ]:
pcas = {}
pca_n_components = {}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, fd in enumerate([1, 2, 3, 4]):
    X_tr, X_val, y_tr, y_val = splits[fd]
    pca = PCA(n_components=0.95, random_state=42)
    pca.fit(X_tr)
    pcas[fd] = pca
    pca_n_components[fd] = pca.n_components_
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    axes[i].plot(range(1, len(pca.explained_variance_ratio_)+1),
                 pca.explained_variance_ratio_, 'o-', label='Explained')
    axes[i].plot(range(1, len(cumvar)+1), cumvar, 's--', label='Cumulative')
    axes[i].axhline(0.95, color='red', ls=':', label='95% threshold')
    axes[i].set_title(f'FD00{fd} — {pca.n_components_} PCs')
    axes[i].set_xlabel('PC'); axes[i].legend(fontsize=7)
plt.suptitle('PCA Explained Variance', fontweight='bold')
plt.tight_layout(); plt.show()
print('PCs selected:', pca_n_components)

---
## 4. Helper Functions — Metrics & Plots

In [ ]:
def compute_metrics(y_true, y_pred, n_params):
    n = len(y_true)
    sse = np.sum((y_true - y_pred) ** 2)
    r2  = r2_score(y_true, y_pred)
    # AIC = n*ln(SSE/n) + 2k
    aic = n * np.log(sse / n + 1e-10) + 2 * n_params
    return {'SSE': round(sse, 2), 'R2': round(r2, 4), 'AIC': round(aic, 2)}


def plot_diagnostics(y_true, y_pred, model_name, fd, ax_res, ax_acf, ax_scatter):
    residuals = y_true - y_pred
    n = len(residuals)

    # Residuals
    ax_res.plot(residuals, '.', markersize=2, alpha=0.5)
    ax_res.axhline(0, color='red', ls='--')
    ax_res.set_title(f'{model_name} | FD00{fd}\nResiduals')
    ax_res.set_xlabel('Sample'); ax_res.set_ylabel('Residual')

    # Autocorrelation of residuals (lag 1..30)
    max_lag = min(30, n - 1)
    lags = range(1, max_lag + 1)
    acf = [np.corrcoef(residuals[:-k], residuals[k:])[0, 1] for k in lags]
    ax_acf.bar(lags, acf)
    ax_acf.axhline(1.96/np.sqrt(n), color='red', ls='--', lw=0.8)
    ax_acf.axhline(-1.96/np.sqrt(n), color='red', ls='--', lw=0.8)
    ax_acf.set_title('ACF of Residuals'); ax_acf.set_xlabel('Lag')

    # Scatter ŷ vs y
    ax_scatter.scatter(y_true, y_pred, s=2, alpha=0.4)
    mn, mx = y_true.min(), y_true.max()
    ax_scatter.plot([mn, mx], [mn, mx], 'r--')
    ax_scatter.set_title('ŷ vs y_test'); ax_scatter.set_xlabel('y_true'); ax_scatter.set_ylabel('ŷ')


results_all = {}  # store all metrics

---
## 5. Model Building
### 5a. OLS — 1st Order (Forward Selection features)

In [ ]:
for fd in [1, 2, 3, 4]:
    X_tr, X_val, y_tr, y_val = splits[fd]
    idx = fwd_features[fd]
    X_tr_s, X_val_s = X_tr[:, idx], X_val[:, idx]

    model = LinearRegression()
    model.fit(X_tr_s, y_tr)
    y_pred = model.predict(X_val_s)

    m = compute_metrics(y_val, y_pred, len(idx) + 1)
    results_all.setdefault(fd, {})['OLS_1st_FWD'] = m
    print(f'FD00{fd} OLS 1st (Forward): {m}')

    fig, axes = plt.subplots(1, 3, figsize=(14, 3))
    plot_diagnostics(y_val, y_pred, 'OLS 1st (Fwd)', fd, *axes)
    plt.tight_layout(); plt.show()

### 5b. OLS — 2nd Order (Forward Selection features)

In [ ]:
for fd in [1, 2, 3, 4]:
    X_tr, X_val, y_tr, y_val = splits[fd]
    idx = fwd_features[fd]
    X_tr_s, X_val_s = X_tr[:, idx], X_val[:, idx]

    poly = PolynomialFeatures(degree=2, include_bias=False)
    X_tr_p  = poly.fit_transform(X_tr_s)
    X_val_p = poly.transform(X_val_s)

    model = LinearRegression()
    model.fit(X_tr_p, y_tr)
    y_pred = model.predict(X_val_p)

    m = compute_metrics(y_val, y_pred, X_tr_p.shape[1] + 1)
    results_all[fd]['OLS_2nd_FWD'] = m
    print(f'FD00{fd} OLS 2nd (Forward): {m}')

    fig, axes = plt.subplots(1, 3, figsize=(14, 3))
    plot_diagnostics(y_val, y_pred, 'OLS 2nd (Fwd)', fd, *axes)
    plt.tight_layout(); plt.show()

### 5c. Lasso Regression (CV-tuned alpha)

In [ ]:
for fd in [1, 2, 3, 4]:
    X_tr, X_val, y_tr, y_val = splits[fd]
    idx = lasso_features[fd]
    X_tr_s, X_val_s = X_tr[:, idx], X_val[:, idx]

    model = LassoCV(cv=5, max_iter=10000, random_state=42)
    model.fit(X_tr_s, y_tr)
    y_pred = model.predict(X_val_s)

    m = compute_metrics(y_val, y_pred, len(idx) + 1)
    results_all[fd]['Lasso'] = m
    print(f'FD00{fd} Lasso (alpha={model.alpha_:.4f}): {m}')

    fig, axes = plt.subplots(1, 3, figsize=(14, 3))
    plot_diagnostics(y_val, y_pred, 'Lasso', fd, *axes)
    plt.tight_layout(); plt.show()

### 5d. Ridge Regression (Bayesian Linear Regression)

In [ ]:
for fd in [1, 2, 3, 4]:
    X_tr, X_val, y_tr, y_val = splits[fd]

    model = RidgeCV(alphas=np.logspace(-3, 4, 50), cv=5)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)

    m = compute_metrics(y_val, y_pred, X_tr.shape[1] + 1)
    results_all[fd]['Ridge'] = m
    print(f'FD00{fd} Ridge (alpha={model.alpha_:.4f}): {m}')

    fig, axes = plt.subplots(1, 3, figsize=(14, 3))
    plot_diagnostics(y_val, y_pred, 'Ridge', fd, *axes)
    plt.tight_layout(); plt.show()

### 5e. Principal Component Regression (PCR)

In [ ]:
for fd in [1, 2, 3, 4]:
    X_tr, X_val, y_tr, y_val = splits[fd]
    pca = pcas[fd]

    X_tr_pca  = pca.transform(X_tr)
    X_val_pca = pca.transform(X_val)

    model = LinearRegression()
    model.fit(X_tr_pca, y_tr)
    y_pred = model.predict(X_val_pca)

    m = compute_metrics(y_val, y_pred, pca.n_components_ + 1)
    results_all[fd]['PCR'] = m
    print(f'FD00{fd} PCR ({pca.n_components_} PCs): {m}')

    fig, axes = plt.subplots(1, 3, figsize=(14, 3))
    plot_diagnostics(y_val, y_pred, 'PCR', fd, *axes)
    plt.tight_layout(); plt.show()

---
## 6. Results Summary Table

In [ ]:
for fd in [1, 2, 3, 4]:
    print(f'\n=== FD00{fd} ===')
    df_res = pd.DataFrame(results_all[fd]).T
    df_res.index.name = 'Model'
    display(df_res.style.highlight_min(subset=['SSE','AIC'], color='lightgreen')
                        .highlight_max(subset=['R2'], color='lightgreen'))

---
## 7. Overall Comparison — Bar Charts (R², SSE, AIC)

In [ ]:
model_names = ['OLS_1st_FWD', 'OLS_2nd_FWD', 'Lasso', 'Ridge', 'PCR']
metrics_to_plot = ['R2', 'SSE', 'AIC']

fig, axes = plt.subplots(3, 4, figsize=(20, 12))

for row, metric in enumerate(metrics_to_plot):
    for col, fd in enumerate([1, 2, 3, 4]):
        vals = [results_all[fd].get(mn, {}).get(metric, np.nan) for mn in model_names]
        axes[row, col].bar(model_names, vals, color=sns.color_palette('Set2', len(model_names)))
        axes[row, col].set_title(f'FD00{fd} — {metric}', fontsize=10)
        axes[row, col].set_xticklabels(model_names, rotation=30, ha='right', fontsize=8)
        axes[row, col].set_ylabel(metric)

plt.suptitle('Model Comparison: R², SSE, AIC across FD001–FD004', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 8. Inference: Autocorrelation Analysis

**Observations:**
- If ACF bars for low lags exceed the significance bounds (±1.96/√n, dashed red), residuals are **autocorrelated** — indicating the model misses temporal patterns in the degradation trajectory.
- OLS 1st order typically shows higher autocorrelation than Ridge/PCR because it cannot capture the nonlinear degradation trend.
- OLS 2nd order reduces autocorrelation but risks overfitting (watch AIC penalty).
- Ridge and PCR tend to have more structurally white residuals on single-condition subsets (FD001/FD003), while multi-condition subsets (FD002/FD004) retain some autocorrelation due to operating-regime shifts not modelled by any regression approach.
- Presence of autocorrelation is a signal that recurrent/time-series methods would outperform static regression for this task — but within the project scope (function approximation), Ridge and PCR offer the best trade-off.

---
## 9. Final Model — Test Set Evaluation (NASA test files)

In [ ]:
print('\n=== Final Model (Ridge) on Official Test Set ===')

for fd in [1, 2, 3, 4]:
    # Retrain Ridge on full training data
    df_tr = datasets[fd]['train']
    df_te = datasets[fd]['test']

    X_full = df_tr[FEATURES].values
    y_full = df_tr['RUL'].values

    scaler = StandardScaler()
    X_full_s = scaler.fit_transform(X_full)

    ridge = RidgeCV(alphas=np.logspace(-3, 4, 50), cv=5)
    ridge.fit(X_full_s, y_full)

    # Evaluate on last cycle of each test engine
    last_test = df_te.groupby('unit').last().reset_index()
    X_test_s = scaler.transform(last_test[FEATURES].values)
    y_test   = last_test['RUL'].values
    y_pred   = ridge.predict(X_test_s)

    m = compute_metrics(y_test, y_pred, X_full_s.shape[1] + 1)
    print(f'FD00{fd} Ridge Test: {m}')

    fig, axes = plt.subplots(1, 3, figsize=(14, 3))
    plot_diagnostics(y_test, y_pred, f'Ridge (Test Set)', fd, *axes)
    plt.tight_layout(); plt.show()